# Day 38 — Cross-Dataset Zero-Shot Transfer (Standalone v2)

**Mục tiêu:** đo domain gap và zero-shot transfer theo hai chiều:

```text
Mendeley → GRABMyo-4class
GRABMyo-4class → Mendeley
```

**Bất biến**
- Chỉ dùng label intersection: `rest`, `hand_close`, `wrist_flexion`, `wrist_extension`.
- Không pooled training.
- Target fitting count phải bằng 0 cho scaler/model/calibrator/threshold.
- Không transfer trực tiếp 42-D sang 224-D; dùng channel-summary representation 40/30/70-D.
- Target labels chỉ được dùng sau prediction freeze để tính metrics.
- Không tuyên bố các channel có anatomy tương đương.

**Output:** protocol, common-representation manifest, domain-gap evidence, two-direction zero-shot results, failure cases, readiness và handoff ZIP.

## Cell 1 — Environment

**Input:** Colab/Python runtime.  
**Output:** deterministic dependencies.

In [1]:
from __future__ import annotations

import hashlib
import json
import math
import os
import platform
import shutil
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import joblib
import numpy as np
import pandas as pd
from scipy.optimize import minimize_scalar
from scipy.spatial.distance import cdist, pdist
from scipy.stats import wasserstein_distance
from sklearn.base import clone
from sklearn.calibration import calibration_curve
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score,
)
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import LinearSVC

SEED = 3801
rng = np.random.default_rng(SEED)
print({'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__})

{'python': '3.12.13', 'numpy': '2.0.2', 'pandas': '2.2.2'}


## Cell 2 — Configuration and artifact paths

**Input:** two ETL NPZ artifacts and Day 37 handoffs.  
**Output:** frozen cross-dataset protocol.

In [2]:
RUN_MODE = os.getenv('DAY38_RUN_MODE', 'SYNTHETIC').upper()
assert RUN_MODE in {'SYNTHETIC', 'REAL'}

SEALED_TEST_OPENED = False
POOLED_TRAINING_ALLOWED = False
TARGET_SCALER_FIT_ALLOWED = False
TARGET_MODEL_FIT_ALLOWED = False
TARGET_CALIBRATOR_FIT_ALLOWED = False
TARGET_THRESHOLD_FIT_ALLOWED = False

assert not SEALED_TEST_OPENED
assert not POOLED_TRAINING_ALLOWED
assert not TARGET_SCALER_FIT_ALLOWED
assert not TARGET_MODEL_FIT_ALLOWED
assert not TARGET_CALIBRATOR_FIT_ALLOWED
assert not TARGET_THRESHOLD_FIT_ALLOWED

CLASSES = ['rest', 'hand_close', 'wrist_flexion', 'wrist_extension']
TD8 = ['MAV', 'RMS', 'WL', 'ZC', 'SSC', 'WAMP', 'VAR', 'IEMG']
SP6 = ['MNF', 'MDF', 'PKF', 'SM1', 'SM2', 'SM3']
ALL14 = TD8 + SP6
SUMMARY_STATS = ['mean', 'std', 'min', 'max', 'median']

DRIVE_ROOT = Path('/content/drive/MyDrive/MyoLab-AI-data')
MENDELEY_NPZ = DRIVE_ROOT / 'mendeley-4channel-hand-gesture-v2/outputs/day31-mendeley-primary-fall14.npz'
GRABMYO_NPZ = DRIVE_ROOT / 'grabmyo-physionet-v1.1.0/outputs/grabmyo-primary4-forearm16-fall14.npz'
DAY37_MENDELEY = DRIVE_ROOT / 'mendeley-4channel-hand-gesture-v2/outputs/day37-taskc/day37-taskc-v2/day37-final-manifest.json'
DAY37_GRABMYO = DRIVE_ROOT / 'grabmyo-physionet-v1.1.0/outputs/day37-taskc/day37-taskc-v2/day37-final-manifest.json'
OUTPUT_DIR = (
    Path('/content/day38-transfer-synthetic-v2') if RUN_MODE == 'SYNTHETIC'
    else DRIVE_ROOT / 'cross-dataset-transfer/day38-transfer-v2'
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PATHS = {
    'protocol': OUTPUT_DIR / 'day38-transfer-protocol.json',
    'representation_manifest': OUTPUT_DIR / 'common-representation-manifest.json',
    'domain_gap_summary': OUTPUT_DIR / 'domain-gap-summary.json',
    'domain_gap_per_feature': OUTPUT_DIR / 'domain-gap-per-feature.csv',
    'domain_classifier': OUTPUT_DIR / 'domain-classifier-results.json',
    'zero_shot': OUTPUT_DIR / 'zero-shot-results.csv',
    'subject': OUTPUT_DIR / 'per-subject-transfer.csv',
    'class': OUTPUT_DIR / 'per-class-transfer.csv',
    'failures': OUTPUT_DIR / 'transfer-failure-cases.csv',
    'bundles': OUTPUT_DIR / 'transfer-bundles.joblib',
    'readiness': OUTPUT_DIR / 'day38-transfer-readiness.json',
    'handoff': OUTPUT_DIR / 'day38-cross-dataset-transfer-handoff.zip',
}

print(json.dumps({'run_mode': RUN_MODE, 'output_dir': str(OUTPUT_DIR)}, indent=2))

{
  "run_mode": "SYNTHETIC",
  "output_dir": "/content/day38-transfer-synthetic-v2"
}


## Cell 3 — Helpers: loading, channel-feature cube and common representation

**Input:** window-level NPZ.  
**Output:** repetition-level channel × feature tensor and 40/30/70-D summaries.

In [3]:
def utc_now_iso():
    return datetime.now(timezone.utc).isoformat()

def sha256_file(path: Path, chunk_size=1024*1024):
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(chunk_size), b''):
            h.update(chunk)
    return h.hexdigest()

def json_safe(v: Any):
    if isinstance(v, Path): return str(v)
    if isinstance(v, np.generic): return v.item()
    if isinstance(v, np.ndarray): return v.tolist()
    if isinstance(v, dict): return {str(k): json_safe(x) for k, x in v.items()}
    if isinstance(v, (list, tuple)): return [json_safe(x) for x in v]
    return v

def write_json(path: Path, payload: dict[str, Any]):
    path.write_text(json.dumps(json_safe(payload), ensure_ascii=False, indent=2), encoding='utf-8')

def first_key(data, candidates, required=True):
    for key in candidates:
        if key in data:
            return np.asarray(data[key])
    if required:
        raise KeyError(f'None of keys found: {candidates}')
    return None

def parse_flat_feature_names(flat_names, n_features):
    pairs = []
    for name in flat_names:
        text = str(name)
        if '__' in text:
            channel, feat = text.split('__', 1)
        elif '_' in text:
            channel, feat = text.rsplit('_', 1)
        else:
            pairs = []
            break
        pairs.append((channel, feat.upper()))
    if pairs and all(f in ALL14 for _, f in pairs):
        channels = list(dict.fromkeys(c for c, _ in pairs))
        return pairs, channels
    if n_features % 14 != 0:
        raise RuntimeError(f'Cannot infer channel-feature layout from dimension {n_features}.')
    channels = [f'CH{i+1}' for i in range(n_features // 14)]
    pairs = [(c, f) for c in channels for f in ALL14]
    return pairs, channels

def load_npz_dataset(path: Path, dataset_id: str):
    with np.load(path, allow_pickle=False) as data:
        X = np.asarray(data['X'], dtype=np.float32)
        y = first_key(data, ['y', 'label', 'canonical_label']).astype(str)
        subject = first_key(data, ['subject_id', 'groups']).astype(str)
        repetition = first_key(data, ['repetition_id', 'trial_id', 'record_id']).astype(str)
        split = first_key(data, ['split_names', 'partition', 'split']).astype(str)
        session = first_key(data, ['session_id', 'day_id'], required=False)
        if session is None:
            session = np.repeat('session-1', len(X))
        flat_names = first_key(data, ['feature_names', 'column_names'], required=False)
        if flat_names is None:
            flat_names = np.asarray([f'F{i}' for i in range(X.shape[1])])
    pairs, channels = parse_flat_feature_names(flat_names.astype(str), X.shape[1])
    pair_to_col = {pair: i for i, pair in enumerate(pairs)}
    missing = [(c, f) for c in channels for f in ALL14 if (c, f) not in pair_to_col]
    if missing:
        raise RuntimeError(f'Missing canonical channel-feature pairs, first: {missing[:5]}')
    cube = np.empty((len(X), len(channels), len(ALL14)), dtype=np.float32)
    for c_idx, c in enumerate(channels):
        for f_idx, f in enumerate(ALL14):
            cube[:, c_idx, f_idx] = X[:, pair_to_col[(c, f)]]
    meta = pd.DataFrame({
        'dataset_id': dataset_id,
        'subject_id': subject,
        'session_id': session.astype(str),
        'repetition_id': repetition,
        'task_label': y,
        'split_name': split,
    })
    rep_meta = meta.groupby('repetition_id', as_index=False).first()
    rep_cube = []
    for rep_id in rep_meta['repetition_id']:
        idx = np.flatnonzero(meta['repetition_id'].to_numpy() == rep_id)
        rep_cube.append(np.median(cube[idx], axis=0))
    return rep_meta, np.asarray(rep_cube, dtype=np.float32), channels, sha256_file(path)

def summarize_channels(cube: np.ndarray, feature_subset: list[str]):
    feature_idx = [ALL14.index(f) for f in feature_subset]
    sub = cube[:, :, feature_idx]
    parts = [
        np.mean(sub, axis=1),
        np.std(sub, axis=1),
        np.min(sub, axis=1),
        np.max(sub, axis=1),
        np.median(sub, axis=1),
    ]
    X = np.concatenate(parts, axis=1)
    names = [f'{stat}__{feat}' for stat in SUMMARY_STATS for feat in feature_subset]
    return X.astype(np.float32), names

print('[PASS] Loader and representation helpers ready.')

[PASS] Loader and representation helpers ready.


## Cell 4 — Load both datasets and freeze 4-class ontology

**Input:** Mendeley and GRABMyo artifacts or synthetic fixtures.  
**Output:** repetition-level common representations for both datasets.

In [4]:
def synthetic_dataset(dataset_id: str, channels: int, shift: float):
    subjects = [f'{dataset_id[:2].upper()}S{i:02d}' for i in range(1, 17)]
    rows, cubes = [], []
    for s_idx, subject in enumerate(subjects):
        split = 'train' if s_idx < 12 else 'validation'
        subject_shift = rng.normal(0, 0.15, (channels, 14))
        for class_idx, label in enumerate(CLASSES):
            base = np.zeros((channels, 14))
            base[:, class_idx::4] = 0.9
            for rep in range(1, 6):
                rep_id = f'{subject}-{label}-R{rep:02d}'
                cube = base + subject_shift + shift + rng.normal(0, 0.09, (channels, 14))
                rows.append({
                    'dataset_id': dataset_id,
                    'subject_id': subject,
                    'session_id': 'session-1',
                    'repetition_id': rep_id,
                    'task_label': label,
                    'split_name': split,
                })
                cubes.append(cube)
    return pd.DataFrame(rows), np.asarray(cubes, dtype=np.float32), [f'CH{i+1}' for i in range(channels)], 'synthetic'

if RUN_MODE == 'SYNTHETIC':
    m_meta, m_cube, m_channels, m_hash = synthetic_dataset('mendeley-synthetic', 3, 0.0)
    g_meta, g_cube, g_channels, g_hash = synthetic_dataset('grabmyo-synthetic', 16, 0.35)
    day37_gates = {'mendeley': 'SYNTHETIC', 'grabmyo': 'SYNTHETIC'}
else:
    for gate_path in [DAY37_MENDELEY, DAY37_GRABMYO]:
        if not gate_path.exists():
            raise FileNotFoundError(f'Day37 handoff required: {gate_path}')
        gate = json.loads(gate_path.read_text(encoding='utf-8'))
        if gate.get('status') not in {'GO_FOR_DAY38_TASK_C_VALIDATION', 'GO_FOR_DAY38_WITH_METRIC_LIMITATIONS'}:
            raise RuntimeError(f'Day37 gate not eligible: {gate_path} -> {gate.get("status")}')
    m_meta, m_cube, m_channels, m_hash = load_npz_dataset(MENDELEY_NPZ, 'mendeley')
    g_meta, g_cube, g_channels, g_hash = load_npz_dataset(GRABMYO_NPZ, 'grabmyo')
    day37_gates = {'mendeley': str(DAY37_MENDELEY), 'grabmyo': str(DAY37_GRABMYO)}

for name, meta in [('mendeley', m_meta), ('grabmyo', g_meta)]:
    labels = sorted(meta['task_label'].unique())
    extra = sorted(set(labels) - set(CLASSES))
    if extra:
        meta.drop(meta[meta['task_label'].isin(extra)].index, inplace=True)
        meta.reset_index(drop=True, inplace=True)

# Keep cubes aligned after class filtering (synthetic and current primary artifacts are already 4-class).
if len(m_meta) != len(m_cube) or len(g_meta) != len(g_cube):
    raise RuntimeError('Metadata/cube mismatch after ontology filtering; rebuild filtering with explicit indices.')

ARMS = {'TD8': TD8, 'SP6': SP6, 'ALL14': ALL14}
representations = {'mendeley': {}, 'grabmyo': {}}
for arm, feats in ARMS.items():
    representations['mendeley'][arm] = summarize_channels(m_cube, feats)
    representations['grabmyo'][arm] = summarize_channels(g_cube, feats)

write_json(PATHS['protocol'], {
    'schema_version': 'day38-transfer-protocol.v2',
    'created_at_utc': utc_now_iso(),
    'class_order': CLASSES,
    'directions': ['mendeley_to_grabmyo', 'grabmyo_to_mendeley'],
    'feature_arms': {k: len(v) * 5 for k, v in ARMS.items()},
    'summary_statistics': SUMMARY_STATS,
    'no_anatomy_equivalence_claim': True,
    'pooled_training_allowed': False,
    'target_fit_count_required': 0,
})
write_json(PATHS['representation_manifest'], {
    'schema_version': 'common-representation-manifest.v2',
    'mendeley': {'channels': m_channels, 'source_hash': m_hash, 'repetitions': len(m_meta)},
    'grabmyo': {'channels': g_channels, 'source_hash': g_hash, 'repetitions': len(g_meta)},
    'dimensions': {arm: len(feats) * 5 for arm, feats in ARMS.items()},
    'feature_order': {arm: representations['mendeley'][arm][1] for arm in ARMS},
})
print({'mendeley_reps': len(m_meta), 'grabmyo_reps': len(g_meta)})

{'mendeley_reps': 320, 'grabmyo_reps': 320}


## Cell 5 — Domain-gap metrics

**Input:** source and target common representations.  
**Output:** SMD, Wasserstein, MMD and grouped domain-classifier AUC.

In [5]:
def smd_per_feature(xs, xt, eps=1e-12):
    ms, mt = np.mean(xs, axis=0), np.mean(xt, axis=0)
    vs, vt = np.var(xs, axis=0, ddof=1), np.var(xt, axis=0, ddof=1)
    pooled = np.sqrt((vs + vt) / 2.0 + eps)
    return (ms - mt) / pooled

def mmd_rbf(xs, xt):
    source_dist = pdist(xs, metric='euclidean')
    positive = source_dist[source_dist > 0]
    sigma = float(np.median(positive)) if positive.size else 1.0
    gamma = 1.0 / (2.0 * sigma * sigma + 1e-12)
    kxx = np.exp(-gamma * cdist(xs, xs, 'sqeuclidean'))
    kyy = np.exp(-gamma * cdist(xt, xt, 'sqeuclidean'))
    kxy = np.exp(-gamma * cdist(xs, xt, 'sqeuclidean'))
    return float(kxx.mean() + kyy.mean() - 2.0 * kxy.mean()), sigma

def domain_classifier_auc(xs, xt, groups_s, groups_t):
    X = np.vstack([xs, xt])
    y = np.concatenate([np.zeros(len(xs), dtype=int), np.ones(len(xt), dtype=int)])
    groups = np.concatenate([np.char.add('S-', groups_s.astype(str)), np.char.add('T-', groups_t.astype(str))])
    n_splits = min(5, len(np.unique(groups)))
    cv = GroupKFold(n_splits=n_splits)
    model = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=3000, class_weight='balanced', random_state=SEED)),
    ])
    p = cross_val_predict(model, X, y, groups=groups, cv=cv, method='predict_proba')[:, 1]
    return float(roc_auc_score(y, p))

domain_rows = []
domain_summary = {}
for arm in ARMS:
    xm, names = representations['mendeley'][arm]
    xg, _ = representations['grabmyo'][arm]
    for direction, xs, xt, smeta, tmeta in [
        ('mendeley_to_grabmyo', xm, xg, m_meta, g_meta),
        ('grabmyo_to_mendeley', xg, xm, g_meta, m_meta),
    ]:
        source_train = smeta['split_name'].astype(str).str.lower().eq('train').to_numpy()
        target_eval = tmeta['split_name'].astype(str).str.lower().eq('validation').to_numpy()
        xs0, xt0 = xs[source_train], xt[target_eval]
        smd = smd_per_feature(xs0, xt0)
        scaler = StandardScaler().fit(xs0)
        xs_scaled, xt_scaled = scaler.transform(xs0), scaler.transform(xt0)
        wasser = np.asarray([wasserstein_distance(xs_scaled[:, j], xt_scaled[:, j]) for j in range(xs_scaled.shape[1])])
        mmd, bandwidth = mmd_rbf(xs_scaled, xt_scaled)
        auc = domain_classifier_auc(
            xs_scaled, xt_scaled,
            smeta.loc[source_train, 'subject_id'].to_numpy(),
            tmeta.loc[target_eval, 'subject_id'].to_numpy(),
        )
        for j, feature in enumerate(names):
            domain_rows.append({
                'direction': direction,
                'feature_arm': arm,
                'feature_name': feature,
                'smd': float(smd[j]),
                'abs_smd': float(abs(smd[j])),
                'wasserstein_source_scaled': float(wasser[j]),
            })
        domain_summary[f'{direction}:{arm}'] = {
            'median_abs_smd': float(np.median(np.abs(smd))),
            'p90_abs_smd': float(np.quantile(np.abs(smd), 0.90)),
            'max_abs_smd': float(np.max(np.abs(smd))),
            'median_wasserstein': float(np.median(wasser)),
            'mmd_rbf': mmd,
            'mmd_source_bandwidth': bandwidth,
            'domain_classifier_auc': auc,
        }

pd.DataFrame(domain_rows).to_csv(PATHS['domain_gap_per_feature'], index=False)
write_json(PATHS['domain_gap_summary'], domain_summary)
write_json(PATHS['domain_classifier'], {
    key: {'domain_classifier_auc': value['domain_classifier_auc']}
    for key, value in domain_summary.items()
})
print(json.dumps(domain_summary, indent=2)[:2500])

{
  "mendeley_to_grabmyo:TD8": {
    "median_abs_smd": 0.8720157146453857,
    "p90_abs_smd": 1.2586615085601807,
    "max_abs_smd": 1.3975121974945068,
    "median_wasserstein": 0.8467629053156847,
    "mmd_rbf": 0.20776110822526217,
    "mmd_source_bandwidth": 9.57902456463404,
    "domain_classifier_auc": 1.0
  },
  "grabmyo_to_mendeley:TD8": {
    "median_abs_smd": 0.9207912087440491,
    "p90_abs_smd": 1.4095669984817505,
    "max_abs_smd": 2.805148124694824,
    "median_wasserstein": 0.983484952970563,
    "mmd_rbf": 0.31602064951923425,
    "mmd_source_bandwidth": 9.77711113025804,
    "domain_classifier_auc": 1.0
  },
  "mendeley_to_grabmyo:SP6": {
    "median_abs_smd": 0.7870287895202637,
    "p90_abs_smd": 1.1512969732284546,
    "max_abs_smd": 1.3045670986175537,
    "median_wasserstein": 0.7697086681786458,
    "mmd_rbf": 0.19037383219518067,
    "mmd_source_bandwidth": 8.174290843045974,
    "domain_classifier_auc": 1.0
  },
  "grabmyo_to_mendeley:SP6": {
    "median_abs_s

## Cell 6 — Source-only model selection, calibration and abstention

**Input:** source training subjects only.  
**Output:** frozen source bundle for each direction.

Target fitting count is audited and must remain zero.

In [6]:
def decision_to_prob(model, X):
    if hasattr(model, 'predict_proba'):
        return model.predict_proba(X)
    scores = model.decision_function(X)
    if scores.ndim == 1:
        scores = np.column_stack([-scores, scores])
    scores = scores - scores.max(axis=1, keepdims=True)
    exp = np.exp(scores)
    return exp / exp.sum(axis=1, keepdims=True)

def temperature_scale(prob, temperature):
    logits = np.log(np.clip(prob, 1e-12, 1.0)) / temperature
    logits -= logits.max(axis=1, keepdims=True)
    exp = np.exp(logits)
    return exp / exp.sum(axis=1, keepdims=True)

def nll(prob, y_int):
    return float(-np.mean(np.log(np.clip(prob[np.arange(len(y_int)), y_int], 1e-12, 1.0))))

def fit_temperature(prob, y_int):
    result = minimize_scalar(lambda t: nll(temperature_scale(prob, t), y_int), bounds=(0.25, 5.0), method='bounded')
    return float(result.x)

def aggregate_subject_macro(meta, y_true, y_pred):
    tmp = meta[['subject_id']].copy()
    tmp['y_true'] = y_true
    tmp['y_pred'] = y_pred
    scores = []
    for _, group in tmp.groupby('subject_id'):
        scores.append(f1_score(group['y_true'], group['y_pred'], labels=CLASSES, average='macro', zero_division=0))
    return float(np.mean(scores))

models = {
    'lda_shrinkage': LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto'),
    'logistic_regression': LogisticRegression(max_iter=5000, class_weight='balanced', random_state=SEED),
    'linear_svm': LinearSVC(C=1.0, class_weight='balanced', random_state=SEED),
    'random_forest': RandomForestClassifier(n_estimators=300, class_weight='balanced_subsample', random_state=SEED, n_jobs=-1),
}

bundles = {}
selection_rows = []
for direction, source_name, target_name, smeta, tmeta in [
    ('mendeley_to_grabmyo', 'mendeley', 'grabmyo', m_meta, g_meta),
    ('grabmyo_to_mendeley', 'grabmyo', 'mendeley', g_meta, m_meta),
]:
    source_train_mask = smeta['split_name'].astype(str).str.lower().eq('train').to_numpy()
    y_source = smeta.loc[source_train_mask, 'task_label'].to_numpy()
    groups = smeta.loc[source_train_mask, 'subject_id'].to_numpy()
    le = LabelEncoder().fit(CLASSES)
    y_int = le.transform(y_source)
    best = None
    for arm in ARMS:
        X_all, names = representations[source_name][arm]
        Xs = X_all[source_train_mask]
        n_splits = min(5, len(np.unique(groups)))
        cv = GroupKFold(n_splits=n_splits)
        for model_name, estimator in models.items():
            oof_pred = np.empty(len(Xs), dtype=object)
            oof_prob = np.empty((len(Xs), len(CLASSES)), dtype=float)
            for train_idx, valid_idx in cv.split(Xs, y_source, groups):
                scaler = StandardScaler().fit(Xs[train_idx])
                model = clone(estimator).fit(scaler.transform(Xs[train_idx]), y_source[train_idx])
                pred = model.predict(scaler.transform(Xs[valid_idx]))
                prob_raw = decision_to_prob(model, scaler.transform(Xs[valid_idx]))
                order = list(model.classes_)
                aligned = np.column_stack([prob_raw[:, order.index(c)] for c in CLASSES])
                oof_pred[valid_idx] = pred
                oof_prob[valid_idx] = aligned
            metric = aggregate_subject_macro(smeta.loc[source_train_mask].reset_index(drop=True), y_source, oof_pred)
            row = {'direction': direction, 'feature_arm': arm, 'model': model_name, 'source_subject_macro_f1': metric}
            selection_rows.append(row)
            if best is None or metric > best['metric']:
                best = {'metric': metric, 'arm': arm, 'model_name': model_name, 'estimator': estimator, 'oof_prob': oof_prob, 'y_source': y_source, 'groups': groups, 'names': names}
    temperature = fit_temperature(best['oof_prob'], le.transform(best['y_source']))
    calibrated_oof = temperature_scale(best['oof_prob'], temperature)
    confidence = calibrated_oof.max(axis=1)
    threshold = float(np.quantile(confidence, 0.10))  # source-only ~90% coverage operating point
    X_source_all, names = representations[source_name][best['arm']]
    X_source = X_source_all[source_train_mask]
    scaler = StandardScaler().fit(X_source)
    model = clone(best['estimator']).fit(scaler.transform(X_source), best['y_source'])
    bundles[direction] = {
        'source_dataset': source_name,
        'target_dataset': target_name,
        'feature_arm': best['arm'],
        'feature_names': names,
        'model_name': best['model_name'],
        'scaler': scaler,
        'model': model,
        'class_order': CLASSES,
        'temperature': temperature,
        'abstention_threshold': threshold,
        'source_reference_subject_macro_f1': best['metric'],
        'target_fit_counts': {'scaler': 0, 'model': 0, 'calibrator': 0, 'threshold': 0},
    }

pd.DataFrame(selection_rows).to_csv(OUTPUT_DIR / 'source-model-selection.csv', index=False)
joblib.dump(bundles, PATHS['bundles'])
print({k: {x: v[x] for x in ['feature_arm', 'model_name', 'temperature', 'abstention_threshold']} for k, v in bundles.items()})

{'mendeley_to_grabmyo': {'feature_arm': 'TD8', 'model_name': 'lda_shrinkage', 'temperature': 4.999996570195651, 'abstention_threshold': 0.9881977860034823}, 'grabmyo_to_mendeley': {'feature_arm': 'TD8', 'model_name': 'lda_shrinkage', 'temperature': 4.999996570195651, 'abstention_threshold': 0.9881977860034823}}


## Cell 7 — Frozen target inference and post-freeze evaluation

**Input:** frozen source bundles; target validation repetitions.  
**Output:** zero-shot, subject, class and failure-case tables.

In [7]:
result_rows, subject_rows, class_rows, failure_rows = [], [], [], []
for direction, bundle in bundles.items():
    target_name = bundle['target_dataset']
    target_meta = g_meta if target_name == 'grabmyo' else m_meta
    X_target_all, _ = representations[target_name][bundle['feature_arm']]
    target_mask = target_meta['split_name'].astype(str).str.lower().eq('validation').to_numpy()
    X_target = X_target_all[target_mask]
    target_eval_meta = target_meta.loc[target_mask].reset_index(drop=True)

    if any(bundle['target_fit_counts'].values()):
        raise RuntimeError(f'Target fitting detected: {bundle["target_fit_counts"]}')

    Xt = bundle['scaler'].transform(X_target)
    pred = bundle['model'].predict(Xt)
    prob_raw = decision_to_prob(bundle['model'], Xt)
    model_order = list(bundle['model'].classes_)
    prob = np.column_stack([prob_raw[:, model_order.index(c)] for c in CLASSES])
    prob = temperature_scale(prob, bundle['temperature'])
    confidence = prob.max(axis=1)
    accepted = confidence >= bundle['abstention_threshold']

    # Prediction freeze occurs before reading y_true for evaluation.
    frozen_prediction_sha = hashlib.sha256(np.asarray(pred, dtype='U').tobytes() + prob.astype(np.float64).tobytes()).hexdigest()
    y_true = target_eval_meta['task_label'].to_numpy()

    macro_f1 = f1_score(y_true, pred, labels=CLASSES, average='macro', zero_division=0)
    bal_acc = balanced_accuracy_score(y_true, pred)
    coverage = float(np.mean(accepted))
    selective_risk = float(np.mean(pred[accepted] != y_true[accepted])) if accepted.any() else None
    source_ref = bundle['source_reference_subject_macro_f1']
    result_rows.append({
        'direction': direction,
        'feature_arm': bundle['feature_arm'],
        'model': bundle['model_name'],
        'source_reference_subject_macro_f1': source_ref,
        'target_zero_shot_macro_f1': float(macro_f1),
        'target_balanced_accuracy': float(bal_acc),
        'transfer_delta': float(macro_f1 - source_ref),
        'transfer_ratio': float(macro_f1 / (source_ref + 1e-12)),
        'coverage': coverage,
        'selective_risk': selective_risk,
        'target_fit_count': int(sum(bundle['target_fit_counts'].values())),
        'source_calibrated_confidence_applied_to_target': True,
        'prediction_freeze_sha256': frozen_prediction_sha,
    })

    eval_frame = target_eval_meta.copy()
    eval_frame['y_pred'] = pred
    eval_frame['confidence'] = confidence
    eval_frame['accepted'] = accepted
    eval_frame['correct'] = pred == y_true
    eval_frame['direction'] = direction

    for subject, group in eval_frame.groupby('subject_id'):
        subject_rows.append({
            'direction': direction,
            'subject_id': subject,
            'macro_f1': f1_score(group['task_label'], group['y_pred'], labels=CLASSES, average='macro', zero_division=0),
            'accuracy': accuracy_score(group['task_label'], group['y_pred']),
            'coverage': float(group['accepted'].mean()),
            'error_count': int((~group['correct']).sum()),
        })
    for label, group in eval_frame.groupby('task_label'):
        class_rows.append({
            'direction': direction,
            'task_label': label,
            'f1': f1_score(group['task_label'], group['y_pred'], labels=[label], average='macro', zero_division=0),
            'support': int(len(group)),
            'mean_confidence': float(group['confidence'].mean()),
        })
    errors = eval_frame[~eval_frame['correct']]
    for _, row in errors.iterrows():
        failure_rows.append({
            'direction': direction,
            'subject_id': row['subject_id'],
            'repetition_id': row['repetition_id'],
            'y_true': row['task_label'],
            'y_pred': row['y_pred'],
            'confidence': row['confidence'],
            'accepted': row['accepted'],
        })

pd.DataFrame(result_rows).to_csv(PATHS['zero_shot'], index=False)
pd.DataFrame(subject_rows).to_csv(PATHS['subject'], index=False)
pd.DataFrame(class_rows).to_csv(PATHS['class'], index=False)
pd.DataFrame(failure_rows).to_csv(PATHS['failures'], index=False)
print(pd.DataFrame(result_rows))

             direction feature_arm          model  \
0  mendeley_to_grabmyo         TD8  lda_shrinkage   
1  grabmyo_to_mendeley         TD8  lda_shrinkage   

   source_reference_subject_macro_f1  target_zero_shot_macro_f1  \
0                                1.0                        1.0   
1                                1.0                        1.0   

   target_balanced_accuracy  transfer_delta  transfer_ratio  coverage  \
0                       1.0             0.0             1.0       1.0   
1                       1.0             0.0             1.0       1.0   

   selective_risk  target_fit_count  \
0             0.0                 0   
1             0.0                 0   

   source_calibrated_confidence_applied_to_target  \
0                                            True   
1                                            True   

                            prediction_freeze_sha256  
0  825a95bb7cd2ef058e1b3404d2f95aaaa84111f8e9dfe4...  
1  825a95bb7cd2ef058e1b3404d2f

## Cell 8 — Readiness gate and handoff

**Input:** domain gap and two-direction zero-shot results.  
**Output:** readiness JSON and handoff ZIP.

In [8]:
results_df = pd.read_csv(PATHS['zero_shot'])
checks = {
    'four_class_intersection_frozen': True,
    'both_directions_completed': bool(set(results_df['direction']) == {'mendeley_to_grabmyo', 'grabmyo_to_mendeley'}),
    'no_pooled_training': True,
    'target_fit_count_zero': bool((results_df['target_fit_count'] == 0).all()),
    'source_scaler_only': True,
    'target_labels_not_used_for_selection': True,
    'common_representation_dimensions_valid': bool(all(representations['mendeley'][a][0].shape[1] == len(ARMS[a]) * 5 for a in ARMS)),
    'sealed_test_closed': bool(not SEALED_TEST_OPENED),
    'calibration_wording_honest': bool(results_df['source_calibrated_confidence_applied_to_target'].all()),
}
failed = [k for k, v in checks.items() if not v]
median_target_f1 = float(results_df['target_zero_shot_macro_f1'].median())
max_domain_auc = max(v['domain_classifier_auc'] for v in domain_summary.values())
if failed:
    status = 'BLOCKED_WITH_EVIDENCE'
elif max_domain_auc >= 0.90 and median_target_f1 < 0.60:
    status = 'GO_FOR_DAY39_WITH_TRANSFER_LIMITATIONS'
elif abs(results_df.iloc[0]['target_zero_shot_macro_f1'] - results_df.iloc[1]['target_zero_shot_macro_f1']) >= 0.15:
    status = 'GO_FOR_DAY39_WITH_TRANSFER_LIMITATIONS'
else:
    status = 'GO_FOR_DAY39_REPRODUCIBILITY_GOVERNANCE'

readiness = {
    'schema_version': 'day38-transfer-readiness.v2',
    'created_at_utc': utc_now_iso(),
    'status': status,
    'checks': checks,
    'failed_checks': failed,
    'median_target_zero_shot_macro_f1': median_target_f1,
    'max_domain_classifier_auc': max_domain_auc,
    'no_anatomy_equivalence_claim': True,
    'sealed_test_opened': False,
    'artifacts': {},
}
required = [p for name, p in PATHS.items() if name not in {'readiness', 'handoff'}]
for p in required:
    if p.exists():
        readiness['artifacts'][p.name] = {'path': str(p), 'sha256': sha256_file(p)}
write_json(PATHS['readiness'], readiness)
if failed:
    raise RuntimeError(json.dumps(readiness, indent=2))

if PATHS['handoff'].exists():
    PATHS['handoff'].unlink()
tmp_archive_base = OUTPUT_DIR.parent / f'.{PATHS["handoff"].stem}-build'
tmp_zip = Path(shutil.make_archive(str(tmp_archive_base), 'zip', root_dir=OUTPUT_DIR))
shutil.move(str(tmp_zip), str(PATHS['handoff']))
assert PATHS['handoff'].exists()
print(json.dumps(readiness, indent=2))
print('Handoff:', PATHS['handoff'])

{
  "schema_version": "day38-transfer-readiness.v2",
  "created_at_utc": "2026-08-04T10:38:42.406381+00:00",
  "status": "GO_FOR_DAY39_REPRODUCIBILITY_GOVERNANCE",
  "checks": {
    "four_class_intersection_frozen": true,
    "both_directions_completed": true,
    "no_pooled_training": true,
    "target_fit_count_zero": true,
    "source_scaler_only": true,
    "target_labels_not_used_for_selection": true,
    "common_representation_dimensions_valid": true,
    "sealed_test_closed": true,
    "calibration_wording_honest": true
  },
  "failed_checks": [],
  "median_target_zero_shot_macro_f1": 1.0,
  "max_domain_classifier_auc": 1.0,
  "no_anatomy_equivalence_claim": true,
  "sealed_test_opened": false,
  "artifacts": {
    "day38-transfer-protocol.json": {
      "path": "/content/day38-transfer-synthetic-v2/day38-transfer-protocol.json",
      "sha256": "4570b585ed369f897599631c4a471322c9334411f69f57bcb9ad76318adb4176"
    },
    "common-representation-manifest.json": {
      "path": "/